The preliminaries

In [6]:
from openai import OpenAI
from scipy.stats import spearmanr
import jsonschema, json
import os
from dotenv import load_dotenv

length_id_batch = 25

#Hidden OpenAI API Key 
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API Key not found! Ensure you have a .env file with OPENAI_API_KEY defined.")

client = OpenAI(api_key=api_key)

#Pull in AIME and GPQA Diamond
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)

# Load the GPQA pool
with open('gpqa_pool.json', 'r') as f:
    gpqa_data = json.load(f)

# Load the GPQA pool
with open('asudit.json', 'r') as f:
    asudit_data = json.load(f)

# Verify the data loaded correctly
print(f"Loaded {len(aime_data)} items from AIME")
print(f"Loaded {len(gpqa_data)} items from GPQA")
print(f"Loaded {len(asudit_data)} items from asudit")

#split case_ids
gpqa_id = asudit_data["case_ids"][:length_id_batch]
aime_id = asudit_data["case_ids"][length_id_batch:]

gpqa_data_filter = {}
gpqa_responses = {}

for case in gpqa_data:
    if case['case_id'] in gpqa_id:
        gpqa_data_filter[case['case_id']] = case
        gpqa_responses[case['case_id']] = {}

aime_data_filter = {}
aime_responses = {}
for case in aime_data:
    if case['case_id'] in aime_id:
        aime_data_filter[case['case_id']] = case
        aime_responses[case['case_id']] = {}

#print(gpqa_id)
#print(aime_id)
print(len(gpqa_data_filter), len(aime_data_filter))

Loaded 196 items from AIME
Loaded 198 items from GPQA
Loaded 4 items from asudit
25 25


1. Builds a prompt and queries the model K = 50 times with temperature=1.0 passed explicitly on every call (do not omit the parameter and rely on the API default). The exact model(s) you call are determined by your axis assignment (see the table in Your Assignment below).

In [17]:
#Axis C mid-vs-frontier: gpt-4.1-mini (zero-shot) [Condition A] vs gpt-4.1 (zero-shot) [Condition B]
import copy
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field
from typing import Literal

#models = ["gpt-4.1-mini", "gpt-4.1"]
#benchmarks = ['gpqa_diamond', 'aime']
#ground_truth = ['correct_letter', 'correct_answer']

# I use pydantic here to prevent the model from reverting to CoT reasoning in the answer,
# instead forcing a zero-shot to the correct answer
# Schema for GPQA (Multiple Choice)
#lass GPQAResponse(BaseModel):
#    answer: Literal["A", "B", "C", "D"]

# Schema for AIME (Integer)
#class AIMEResponse(BaseModel):
#    answer: int = Field(ge=0, le=999) # Ensures the integer is between 0 and 999

def generate_gpqa_prompt(subject):
    """
    Zero-shot GPQA prompt with a parseable final-answer marker.
    The prompt asks for a brief solution summary, not step-by-step CoT.
    """
    return f"""You are answering a graduate-level multiple-choice question in {subject}.

Rules:
- Single-agent only.
- Do not use tools, web search, external retrieval, or multi-turn interaction.
- Choose exactly one answer choice: A, B, C, or D.
- Provide a brief solution summary, not a detailed step-by-step derivation.
- The final line must have exactly this form:

FINAL_ANSWER: <A, B, C, or D>

Required output format:
SOLUTION_SUMMARY:
<2-5 concise sentences explaining the basis for your answer>

FINAL_ANSWER: <one capital letter: A, B, C, or D>

Do not put anything after the FINAL_ANSWER line."""

system_prompt_aime = """You are solving a math problem from the American Invitational Mathematics Examination.

Rules:
- Single-agent only.
- Do not use tools, web search, external retrieval, or multi-turn interaction.
- The answer must be an integer from 0 to 999.
- Provide a brief solution summary, not a detailed step-by-step derivation.
- The final line must have exactly this form:

FINAL_ANSWER: <integer from 0 to 999>

Required output format:
SOLUTION_SUMMARY:
<2-5 concise sentences explaining the basis for your answer>

FINAL_ANSWER: <one integer from 0 to 999>

Do not put anything after the FINAL_ANSWER line."""

import re

FINAL_ANSWER_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*(?P<answer>[^\r\n]*)[ \t]*$"
)

def extract_final_answer_line(raw_text):
    """
    Extracts the answer text from the last FINAL_ANSWER line.
    Returns None if no such line exists.
    """
    if not isinstance(raw_text, str):
        return None

    matches = list(FINAL_ANSWER_RE.finditer(raw_text))

    if not matches:
        return None

    return matches[-1].group("answer").strip()

def parse_gpqa_response(raw_text):
    """
    Parse GPQA response into "A", "B", "C", "D", or "_UNPARSEABLE_".
    """
    candidate = extract_final_answer_line(raw_text)

    if candidate is None:
        candidate = raw_text.strip()

    candidate = candidate.strip().upper()

    match = re.fullmatch(r"([ABCD])\.?", candidate)

    if match:
        return match.group(1)

    return "_UNPARSEABLE_"


def parse_aime_response(raw_text):
    """
    Parse AIME response into an int from 0 to 999, or "_UNPARSEABLE_".
    """
    candidate = extract_final_answer_line(raw_text)

    if candidate is None:
        candidate = raw_text.strip()

    candidate = candidate.strip()

    match = re.fullmatch(r"(\d{1,3})\.?", candidate)

    if not match:
        return "_UNPARSEABLE_"

    value = int(match.group(1))

    if 0 <= value <= 999:
        return value

    return "_UNPARSEABLE_"


def parse_response(raw_text, benchmark):
    if benchmark == "gpqa":
        return parse_gpqa_response(raw_text)
    elif benchmark == "aime":
        return parse_aime_response(raw_text)
    else:
        raise ValueError(f"Unknown benchmark: {benchmark}")

import time
import random
#from concurrent.futures import ThreadPoolExecutor

def run_agent(system_prompt,
              model,
              user_message,
              case_id,
              repository,
              benchmark,
              iterations=50,
              max_workers=10,
              max_retries=4):
    '''
    Runs K independent samples for one case.

    Stores:
      - answers: parsed answers used for metrics
      - raw_responses: full raw model text, preserving rationale
      - sample_records: metadata useful for debugging
    '''
    
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},]
    
    repository[case_id]["answers"] = []
    repository[case_id]["raw_responses"] = []
    repository[case_id]["sample_records"] = []
    
   
    
    def get_completion(sample_index):
        for attempt in range(max_retries + 1):
            try:
                resp = client.chat.completions.create(
                    model=model, 
                    messages=messages, 
                    max_tokens=700, 
                    temperature=1.0
                )
                
                choice = resp.choices[0]
                raw_text = choice.message.content
                parsed_answer = parse_response(raw_text, benchmark)
                
                return {
                    "sample_index": sample_index,
                    "raw_response": raw_text,
                    "parsed_answer": parsed_answer,
                    "finish_reason": choice.finish_reason,
                    "api_error": None,
                }
            
            except Exception as e:
                if attempt < max_retries:
                    sleep_time = (2 ** attempt) + random.random()
                    time.sleep(sleep_time)
                else:
                    # recommend failing instead of silently counting API failures
                    # as "_UNPARSEABLE_".
                    raise RuntimeError(
                        f"API failed for case_id={case_id}, "
                        f"model={model}, sample_index={sample_index}"
                    ) from e
    
    # Use ThreadPoolExecutor to run iterations in parallel
    # max_workers=20 is a safe starting point to avoid hitting rate limits too fast
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        sample_records = list(executor.map(get_completion, range(iterations)))

    repository[case_id]["sample_records"] = sample_records
    repository[case_id]["raw_responses"] = [
            r["raw_response"] for r in sample_records
    ]
    repository[case_id]["answers"] = [
        r["parsed_answer"] for r in sample_records
    ]

    print(
        f"Completed {iterations} iterations for case {case_id} "
        f"using {model}; "
        f"unparseable={repository[case_id]['answers'].count('_UNPARSEABLE_')}"
    )
    
   
        
       
gpqa_repo = {"gpt-4.1-mini":copy.deepcopy(gpqa_responses), "gpt-4.1":copy.deepcopy(gpqa_responses)}
models = ["gpt-4.1-mini", "gpt-4.1"]

def format_gpqa_user_message(case):
    """
    Build the full GPQA user message.

    If choices are stored separately in case["options"], include them.
    If the question already contains labeled choices, return it as-is.
    """
    question = case["question"].strip()

    options = case.get("options")

    if isinstance(options, dict):
        option_lines = []

        for letter in ["A", "B", "C", "D"]:
            if letter not in options:
                raise ValueError(
                    f"Missing option {letter} for case_id={case.get('case_id')}"
                )

            option_lines.append(f"{letter}. {options[letter]}")

        option_text = "\n".join(option_lines)

        return f"""Question:
{question}

Answer choices:
{option_text}"""

    

for model in gpqa_repo.keys():
    for case_id in gpqa_id:
        case = gpqa_data_filter[case_id]
        system_prompt_gpqa = generate_gpqa_prompt(case['subject'])
        
        run_agent(system_prompt=system_prompt_gpqa,
                  model=model,
                  user_message=format_gpqa_user_message(case),
                  case_id=case_id,
                  repository=gpqa_repo[model],
                  benchmark="gpqa",
                  iterations=50,
                  max_workers=10)

    # Save after all 25 GPQA cases for this model are done
    with open(f"raw_gpqa_{model}_results.json", "w") as f:
        json.dump(gpqa_repo[model], f, indent=4)

aime_repo = {"gpt-4.1-mini":copy.deepcopy(aime_responses), "gpt-4.1":copy.deepcopy(aime_responses)}
for model in aime_repo.keys():
    for case_id in aime_id:
        #system_prompt_gpqa = generate_gpqa_prompt(gpqa_data_filter[id]['subject'])
        run_agent(system_prompt=system_prompt_aime,
                  model=model,
                  user_message=case["question"],
                  case_id=case_id,
                  repository=aime_repo[model],
                  benchmark="aime",
                  iterations=50,
                  max_workers=10)
    
    # Save after all 25 AIME cases for this model are done
    with open(f"raw_aime_{model}_results.json", "w") as f:
        json.dump(aime_repo[model], f, indent=4)





Completed 50 iterations for case gpqa_d_157 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_151 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_032 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_096 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_101 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_015 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_072 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_002 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_113 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_075 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_156 using gpt-4.1-mini; unparseable=0
Completed 50 iterations for case gpqa_d_016 using gpt-4.1-mini; unparseable=1
Completed 50 iterations for case gpqa_d_165 using gpt-4.1-mini; 

2. We already parsed the output while calling the OpenAI API in task 1, so the focus of this code block is QA/QC of the data. 

In [18]:
import pandas as pd

dataset_names = ["gpqa", "aime"]
models = ["gpt-4.1-mini", "gpt-4.1"]
master_rows = []
condition_for_model = {
    "gpt-4.1-mini": "a",
    "gpt-4.1": "b",
}

for model in models:
    for dataset in dataset_names:
        with open(f'raw_{dataset}_{model}_results.json', 'r') as f:
            data = json.load(f)

        for case_id, info in data.items():
            # Add metadata so I can group/filter later
            row = dict(info)
            row["case_id"] = case_id
            row["model_name"] = model   
            row["benchmark"] = dataset
            row["condition"] = condition_for_model[model]
            master_rows.append(row)
            

# Create one single DF for all 100 entries
full_df = pd.DataFrame(master_rows)

print(f"Total rows: {len(full_df)}")

if len(full_df) != 100:
    print(f"ERROR: Expected 100 rows, found {len(full_df)}")

required_columns = {
    "case_id",
    "model_name",
    "benchmark",
    "condition",
    "answers",
}

missing_columns = required_columns - set(full_df.columns)

if missing_columns:
    print(f"ERROR: Missing required columns: {missing_columns}")
else:
    print("All required columns are present.")

#Run Quality Control Checks for Parsing Task
# (a) Check length of answers list in every row
full_df['sample_count'] = full_df['answers'].apply(len)

# Flag specific case_ids that failed to reach K=50
if (full_df['sample_count'] != 50).any():
    print("\nERROR: Missing samples detected in these cases:")
    print(full_df[full_df['sample_count'] != 50][['case_id', 'model_name', 'sample_count']])
else:
    print("\nAll rows have exactly K=50 samples.")

if "raw_responses" in full_df.columns:
    full_df["raw_response_count"] = full_df["raw_responses"].apply(len)

    if (full_df["raw_response_count"] != 50).any():
        print("\nERROR: Missing raw responses detected in these cases:")
        print(
            full_df[
                full_df["raw_response_count"] != 50
            ][["case_id", "model_name", "benchmark", "raw_response_count"]]
        )
    else:
        print("\nAll rows have exactly K=50 raw responses.")

if "sample_records" in full_df.columns:
    full_df["sample_record_count"] = full_df["sample_records"].apply(len)

    if (full_df["sample_record_count"] != 50).any():
        print("\nERROR: Missing sample records detected in these cases:")
        print(
            full_df[
                full_df["sample_record_count"] != 50
            ][["case_id", "model_name", "benchmark", "sample_record_count"]]
        )
    else:
        print("\nAll rows have exactly K=50 sample records.")

# (b) Verify experimental balance
print("\n--- Experimental Balance Check ---")

# Check Model distribution: Should be 50 'gpt-4.1-mini' and 50 'gpt-4.1'
print("Counts by Model Name:")
print(full_df['model_name'].value_counts())

# Check Condition distribution: Should be 50 'a' and 50 'b'
print("\nCounts by Condition Label:")
print(full_df['condition'].value_counts())

# Check Benchmark distribution: Should be 50 'gpqa' and 50 'aime'
print("\nCounts by Benchmark Dataset:")
print(full_df['benchmark'].value_counts())

# Cross-tabulation: Each cell in this 2x2 matrix must be exactly 25
print("\nSegment Balance (Expected: 25 per cell):")
segment_check = pd.crosstab(full_df['model_name'], full_df['benchmark'])
print(segment_check)

expected_segment_counts = pd.DataFrame(
    25,
    index=models,
    columns=dataset_names,
)

segment_check = segment_check.reindex(index=models, columns=dataset_names)

print("\nSegment Balance Expected:")
print(expected_segment_counts)

print("\nSegment Balance Observed:")
print(segment_check)

if not segment_check.equals(expected_segment_counts):
    print("\nERROR: Segment balance is not 25 per model/benchmark cell.")
else:
    print("\nSegment balance is correct.")

#Verify the Output of the Answers

# --- GPQA SECTION ---
valid_gpqa_set = {'A', 'B', 'C', 'D', '_UNPARSEABLE_'}

def find_invalid_gpqa(answers):
    return [a for a in answers if a not in valid_gpqa_set]

# 1. Isolate
gpqa_only = full_df[full_df['benchmark'] == 'gpqa'].copy()

# 2. Apply function DIRECTLY to the slice
gpqa_only['invalid_gpqa'] = gpqa_only['answers'].apply(find_invalid_gpqa)

# 3. Check length (Safe because there are no AIME NaNs in gpqa_only)
gpqa_errors = gpqa_only[gpqa_only['invalid_gpqa'].map(len) > 0]
print(f"GPQA Rows with invalid sample values: {len(gpqa_errors)}")


# --- AIME SECTION ---
def find_invalid_aime(answers):
    invalid_found = []
    for a in answers:
        if a == '_UNPARSEABLE_':
            continue
        if not isinstance(a, int) or not (0 <= a <= 999):
            invalid_found.append(a)
    return invalid_found

# 1. Isolate
aime_only = full_df[full_df['benchmark'] == 'aime'].copy()

# 2. Apply function DIRECTLY to the slice
aime_only['invalid_aime'] = aime_only['answers'].apply(find_invalid_aime)

# 3. Check length (Safe because there are no GPQA NaNs in aime_only)
aime_errors = aime_only[aime_only['invalid_aime'].map(len) > 0]
print(f"AIME Rows with invalid sample values: {len(aime_errors)}")

def count_unparseable(answers):
    return sum(a == "_UNPARSEABLE_" for a in answers)

full_df["unparseable_count"] = full_df["answers"].apply(count_unparseable)
full_df["unparseable_rate"] = full_df["unparseable_count"] / full_df["sample_count"]

print("\nUnparseable Summary:")
print(
    full_df.groupby(["benchmark", "model_name"])["unparseable_count"]
    .agg(["sum", "mean", "max"])
)

Total rows: 100
All required columns are present.

All rows have exactly K=50 samples.

All rows have exactly K=50 raw responses.

All rows have exactly K=50 sample records.

--- Experimental Balance Check ---
Counts by Model Name:
model_name
gpt-4.1-mini    50
gpt-4.1         50
Name: count, dtype: int64

Counts by Condition Label:
condition
a    50
b    50
Name: count, dtype: int64

Counts by Benchmark Dataset:
benchmark
gpqa    50
aime    50
Name: count, dtype: int64

Segment Balance (Expected: 25 per cell):
benchmark     aime  gpqa
model_name              
gpt-4.1         25    25
gpt-4.1-mini    25    25

Segment Balance Expected:
              gpqa  aime
gpt-4.1-mini    25    25
gpt-4.1         25    25

Segment Balance Observed:
benchmark     gpqa  aime
model_name              
gpt-4.1-mini    25    25
gpt-4.1         25    25

Segment balance is correct.
GPQA Rows with invalid sample values: 0
AIME Rows with invalid sample values: 0

Unparseable Summary:
                       

3. Computes per-case metrics — consistency C, accuracy A, risk-adjusted accuracy 𝑅𝛼, and the Wilson lower bound R_wilson (formulas in The Data → Metrics below).

In [32]:
#Create DataFrames of "Ground Truth"
master_rows = []

for dataset in dataset_names:
    with open(f'{dataset}_pool.json', 'r') as f:
        data = json.load(f)

    for case in data:
            # Add metadata so I can group/filter later
        row = dict(case)
        #row["case_id"] = case_id
           
        row["benchmark"] = dataset
        if dataset == "aime":
            row["ground_truth"] = row.pop("correct_answer")
        else:
            row["ground_truth"] = row.pop("correct_letter")
        master_rows.append(row)
            
truth_df = pd.DataFrame(master_rows)[["benchmark", "case_id", "ground_truth"]]

merged_df = pd.merge(
    left=full_df,
    right=truth_df,
    on=["benchmark", "case_id"],
    how="left",
    validate="many_to_one"
)

if merged_df["ground_truth"].isna().any():
    print("ERROR: Some rows are missing ground truth.")
    print(merged_df[merged_df["ground_truth"].isna()][["benchmark", "case_id", "model_name"]])
else:
    print("All rows have ground truth.")

from scipy import stats

from collections import Counter
import numpy as np

K_EXPECTED = 50

def get_majority_answer_and_count(answers, benchmark):
    """
    Returns the majority answer and majority count.

    GPQA:
      - Answers are strings.
      - Tie-break by lexicographically smallest tied answer.

    AIME:
      - Integer answers tie-break by smallest integer.
      - If _UNPARSEABLE_ is strictly most common, it can be majority.
      - If _UNPARSEABLE_ ties with integer answers, prefer the smallest integer.
    """
    counts = Counter(answers)
    max_count = max(counts.values())
    tied = [answer for answer, count in counts.items() if count == max_count]

    if benchmark == "gpqa":
        majority_answer = min(tied)
    elif benchmark == "aime":
        tied_ints = [answer for answer in tied if isinstance(answer, int)]

        if tied_ints:
            majority_answer = min(tied_ints)
        else:
            majority_answer = "_UNPARSEABLE_"
    else:
        raise ValueError(f"Unknown benchmark: {benchmark}")

    return majority_answer, max_count


def count_correct_answers(answers, ground_truth):
    return sum(answer == ground_truth for answer in answers)


def make_answer_counts(answers, benchmark):
    """
    Creates answer_counts in a JSON-friendly way.

    For GPQA, include all possible answer keys.
    For AIME, stringify integer keys because JSON object keys must be strings.
    """
    counts = Counter(answers)

    if benchmark == "gpqa":
        return {
            "A": int(counts.get("A", 0)),
            "B": int(counts.get("B", 0)),
            "C": int(counts.get("C", 0)),
            "D": int(counts.get("D", 0)),
            "_UNPARSEABLE_": int(counts.get("_UNPARSEABLE_", 0)),
        }

    if benchmark == "aime":
        integer_answers = sorted(
            answer for answer in counts
            if isinstance(answer, int)
        )

        answer_counts = {
            str(answer): int(counts[answer])
            for answer in integer_answers
        }

        if counts.get("_UNPARSEABLE_", 0) > 0:
            answer_counts["_UNPARSEABLE_"] = int(counts["_UNPARSEABLE_"])

        return answer_counts

    raise ValueError(f"Unknown benchmark: {benchmark}")

# Number of samples
merged_df["K"] = merged_df["answers"].apply(len)

if (merged_df["K"] != K_EXPECTED).any():
    print("ERROR: Some rows do not have K=50 samples.")
    print(merged_df[merged_df["K"] != K_EXPECTED][["benchmark", "case_id", "model_name", "K"]])
else:
    print("All rows have K=50 samples.")

# Majority answer and consistency
majority_info = merged_df.apply(
    lambda row: get_majority_answer_and_count(row["answers"], row["benchmark"]),
    axis=1
)

merged_df[["majority_answer", "n_majority"]] = pd.DataFrame(
    majority_info.tolist(),
    index=merged_df.index
)

merged_df["C"] = merged_df["n_majority"] / merged_df["K"]

# Accuracy
merged_df["n_correct"] = merged_df.apply(
    lambda row: count_correct_answers(row["answers"], row["ground_truth"]),
    axis=1
)

merged_df["A"] = merged_df["n_correct"] / merged_df["K"]

# Answer counts
merged_df["answer_counts"] = merged_df.apply(
    lambda row: make_answer_counts(row["answers"], row["benchmark"]),
    axis=1
)

for alpha in [0, 1, 2]:
    penalty = alpha * np.sqrt(merged_df["A"] * (1 - merged_df["A"]))
    merged_df[f"R_{alpha}"] = np.maximum(0, merged_df["A"] - penalty)

z = 1.96
n = merged_df["K"]
A = merged_df["A"]

merged_df["R_wilson"] = (
    A
    + z**2 / (2 * n)
    - z * np.sqrt((A * (1 - A) / n) + (z**2 / (4 * n**2)))
) / (
    1 + z**2 / n
)

metric_cols = ["C", "A", "R_0", "R_1", "R_2", "R_wilson"]

merged_df[metric_cols] = merged_df[metric_cols].round(4)

# answer_counts should sum to K
merged_df["answer_count_sum"] = merged_df["answer_counts"].apply(lambda d: sum(d.values()))

bad_answer_counts = merged_df[merged_df["answer_count_sum"] != merged_df["K"]]

if len(bad_answer_counts) > 0:
    print("ERROR: Some answer_counts do not sum to K.")
    print(
        bad_answer_counts[
            ["benchmark", "case_id", "model_name", "K", "answer_count_sum", "answer_counts"]
        ]
    )
else:
    print("All answer_counts sum to K.")

# R_0 should equal A
if not np.allclose(merged_df["R_0"], merged_df["A"]):
    print("ERROR: R_0 does not equal A.")
else:
    print("R_0 equals A.")

# Metrics should be in [0, 1]
for col in metric_cols:
    bad = merged_df[~merged_df[col].between(0, 1)]

    if len(bad) > 0:
        print(f"ERROR: {col} has values outside [0, 1].")
        print(bad[["benchmark", "case_id", "model_name", col]])

'''
#CALCULATE MAJORITY
# 1. Convert your column of lists into a single 2D NumPy array
# This works best if all elements are the same type (e.g., all strings or all ints)
answers_matrix = np.array(merged_df['answers'].tolist())
# 2. Use scipy's mode function (very fast)
# axis=1 tells it to find the mode for each row
mode_result = stats.mode(answers_matrix, axis=1, keepdims=True)
# 3. Extract the values back into your DataFrame
merged_df['majority_answer'] = mode_result.mode.flatten()

#CALCULATE CONSISTENCY
# 1. 'Explode' the answers column
# This turns each row with 50 answers into 50 separate rows
exploded = merged_df[['majority_answer', 'answers']].explode('answers')
# 2. Perform a vectorized comparison
# This creates a boolean Series in one shot
matches = (exploded['answers'] == exploded['majority_answer'])
# 3. Group by the original index and calculate the mean
# This gives you the consistency (C) for every row simultaneously
merged_df['consistency'] = matches.groupby(level=0).mean()

#CALCULATE ACCURACY
exploded = merged_df[['ground_truth', 'answers']].explode('answers')
matches = (exploded['answers'] == exploded['ground_truth'])
merged_df['accuracy'] = matches.groupby(level=0).mean()

#CALCULATE Risk-Adjusted Accuracy
alphas = [0, 1, 2] # Your three values of alpha
for a in alphas:
    # Calculate the standard error part
    # accuracy * (1 - accuracy)
    se = np.sqrt(merged_df['accuracy'] * (1 - merged_df['accuracy']))
    
    # Calculate R_alpha
    r_val = merged_df['accuracy'] - (a * se)
    
    # Use np.clip to handle the max(0, ...) requirement
    merged_df[f'r_alpha_{a}'] = np.clip(r_val, 0, None)

#Calculate R_wilson
z = 1.96
n = 50
k = 50

sq_rt_term = np.sqrt(merged_df["accuracy"] * (1 - merged_df["accuracy"]) / n + z**2/(4*n**2))
numerator = merged_df['accuracy'] + z**2/(2*n) - z * sq_rt_term
denom = 1 + z**2/n
merged_df["R_wilson"] = numerator / denom
'''

All rows have ground truth.
All rows have K=50 samples.
All answer_counts sum to K.
R_0 equals A.


'\n#CALCULATE MAJORITY\n# 1. Convert your column of lists into a single 2D NumPy array\n# This works best if all elements are the same type (e.g., all strings or all ints)\nanswers_matrix = np.array(merged_df[\'answers\'].tolist())\n# 2. Use scipy\'s mode function (very fast)\n# axis=1 tells it to find the mode for each row\nmode_result = stats.mode(answers_matrix, axis=1, keepdims=True)\n# 3. Extract the values back into your DataFrame\nmerged_df[\'majority_answer\'] = mode_result.mode.flatten()\n\n#CALCULATE CONSISTENCY\n# 1. \'Explode\' the answers column\n# This turns each row with 50 answers into 50 separate rows\nexploded = merged_df[[\'majority_answer\', \'answers\']].explode(\'answers\')\n# 2. Perform a vectorized comparison\n# This creates a boolean Series in one shot\nmatches = (exploded[\'answers\'] == exploded[\'majority_answer\'])\n# 3. Group by the original index and calculate the mean\n# This gives you the consistency (C) for every row simultaneously\nmerged_df[\'consist

4. Validates and serializes the results into a single {PennKey}_results.json (for example, dkaihua_results.json) that satisfies the provided JSON schema, then computes per-cell Spearman for the report.